### Tensorflow

In [1]:
import keras

In [2]:
keras.backend.backend()

'tensorflow'

In [2]:
def get_mnist_model():
    inputs = keras.Input(shape=(28 * 28,))
    features = keras.layers.Dense(512, activation="relu")(inputs)
    features = keras.layers.Dropout(0.5)(features)
    outputs = keras.layers.Dense(10, activation="softmax")(features)
    model = keras.Model(inputs, outputs)
    return model

In [3]:
(images, labels), (test_images, test_labels) = keras.datasets.mnist.load_data()

images = images.reshape((60000, 28 * 28)).astype("float32") / 255
test_images = test_images.reshape((10000, 28 * 28)).astype("float32") / 255

train_images, val_images = images[10000:], images[:10000]
train_labels, val_labels = labels[10000:], labels[:10000]


In [7]:
model = get_mnist_model()
loss_fn = keras.losses.SparseCategoricalCrossentropy()
optimizer = keras.optimizers.Adam()

# Tensorflow
def train_step(inputs, targets):
    with tf.GradientTape() as tape:
        predictions = model(inputs, training=True)
        loss = loss_fn(targets, predictions)
    gradients = tape.gradient(loss, model.trainable_weights)
    optimizer.apply(gradients, model.trainable_weights)
    return loss

batch_size = 32
inputs = train_images[:batch_size]
targets = train_labels[:batch_size]
loss = train_step(inputs, targets)
loss.numpy().item()

2.550929307937622

### Torch

In [1]:
# Restart Terminal and run from here
import os
os.environ["KERAS_BACKEND"] = "torch" # torch , tensorflow, jax

import keras
keras.backend.backend()

'torch'

In [6]:
# Torch
model = get_mnist_model()
loss_fn = keras.losses.SparseCategoricalCrossentropy()
optimizer = keras.optimizers.Adam()

def train_step(inputs, targets):
    predictions = model(inputs, training=True)
    loss = loss_fn(targets, predictions)
    loss.backward()
    gradients = [weight.value.grad for weight in model.trainable_weights]
    with torch.no_grad():
        optimizer.apply(gradients, model.trainable_weights)
    model.zero_grad()
    return loss


batch_size = 32
inputs = train_images[:batch_size]
targets = train_labels[:batch_size]
loss = train_step(inputs, targets)
loss

tensor(2.4365, grad_fn=<WhereBackward0>)

### JAX

In [5]:
# Restart Terminal and run from here
import os
os.environ["KERAS_BACKEND"] = "jax" # torch , tensorflow, jax

import keras
import jax
keras.backend.backend()

'jax'

In [4]:
model = get_mnist_model()
loss_fn = keras.losses.SparseCategoricalCrossentropy()

def compute_loss_and_updates(trainable_variables, non_trainable_variables, inputs, targets):
    outputs, non_trainable_variables = model.stateless_call(trainable_variables, non_trainable_variables, inputs,training=True)
    loss = loss_fn(targets, outputs)
    return loss, non_trainable_variables

In [6]:
grad_fn = jax.value_and_grad(compute_loss_and_updates, has_aux=True)


In [7]:
optimizer = keras.optimizers.Adam()
optimizer.build(model.trainable_variables)

def train_step(state, inputs, targets):
    (trainable_variables, non_trainable_variables, optimizer_variables) = state
    (loss, non_trainable_variables), grads = grad_fn(trainable_variables, non_trainable_variables, inputs, targets)
    trainable_variables, optimizer_variables = optimizer.stateless_apply(optimizer_variables, grads, trainable_variables)
    return loss, (
        trainable_variables,
        non_trainable_variables,
        optimizer_variables,
    )

In [10]:
batch_size = 32
inputs = train_images[:batch_size]
targets = train_labels[:batch_size]
trainable_variables = [v.value for v in model.trainable_variables]
non_trainable_variables = [v.value for v in model.non_trainable_variables]
optimizer_variables = [v.value for v in optimizer.variables]
state = (trainable_variables, non_trainable_variables, optimizer_variables)
loss, state = train_step(state, inputs, targets)
loss.item()

2.45222544670105